In [ ]:
import dlt
from pyspark.sql.functions import *

In [ ]:
@dlt.table(
    name="bronze_orders"
)

def bronze_orders():
  data= [
  (301,101,201,"2024-04-01",20,"Delivered"),
  (302,102,201,"2024-04-01",35,"Delivered"),
  (303,111,204,"2024-04-02",2,"Delivered"),
  (304,114,208,"2024-04-02",5,"Pending"),
  (305,115,204,"2024-04-03",3,"Delivered"),
  (306,104,202,"2024-04-03",50,"Delivered"),
  (307,105,202,"2024-04-04",18,"Cancelled"),
  (308,117,206,"2024-04-04",7,"Delivered"),
  (309,118,210,"2024-04-05",4,"Pending"),
  (310,119,206,"2024-04-05",12,"Delivered"),
  (311,120,210,"2024-04-06",6,"Delivered"),
  (312,113,204,"2024-04-06",4,"Delivered"),
  (313,116,208,"2024-04-07",2,"Pending"),
  (314,109,205,"2024-04-07",80,"Delivered"),
  (315,110,205,"2024-04-08",120,"Delivered"),
  (316,106,203,"2024-04-08",60,"Cancelled"),
  (317,107,209,"2024-04-09",25,"Delivered"),
  (318,108,203,"2024-04-09",40,"Delivered"),
  (319,112,208,"2024-04-10",2,"Pending"),
  (320,101,207,"2024-04-10",15,"Delivered")
  ]
  columns = [
  "order_id",
  "product_id",
  "supplier_id",
  "order_date",
  "quantity",
  "order_status"
  ]

  return spark.createDataFrame(data, columns)

In [ ]:
@dlt.table(
    name = "bronze_payments"
)

def bronze_payments():
  payments_data = [
  (401,301,24000,"UPI","Paid"),
  (402,302,31500,"Credit Card","Paid"),
  (403,303,90000,"Bank Transfer","Paid"),
  (404,304,125000,"UPI","Pending"),
  (405,305,186000,"Bank Transfer","Paid"),
  (406,306,3000,"Cash","Paid"),
  (407,307,8100,"UPI","Cancelled"),
  (408,308,24500,"Debit Card","Paid"),
  (409,309,48000,"UPI","Pending"),
  (410,310,33600,"Cash","Paid"),
  (411,311,33000,"Credit Card","Paid"),
  (412,312,116000,"Bank Transfer","Paid"),
  (413,313,84000,"UPI","Pending"),
  (414,314,6000,"Cash","Paid"),
  (415,315,13200,"UPI","Paid"),
  (416,316,7200,"Cash","Cancelled"),
  (417,317,8000,"UPI","Paid"),
  (418,318,3600,"Debit Card","Paid"),
  (419,319,76000,"Bank Transfer","Pending"),
  (420,320,18000,"UPI","Paid")
  ]
  payments_columns = [
  "payment_id",
  "order_id",
  "bill_amount",
  "payment_mode",
  "payment_status"
  ]

  return spark.createDataFrame(payments_data, payments_columns)

In [ ]:
@dlt.table(
    name = "bronze_suppliers"
)

def bronze_suppliers():
  suppliers_data = [
  (201,"Reddy Traders","Hyderabad","Groceries"),
  (202,"Fresh Dairy Ltd","Chennai","Dairy"),
  (203,"CarePlus Suppliers","Mumbai","Personal Care"),
  (204,"Elite Electronics","Delhi","Electronics"),
  (205,"OfficeKart","Bengaluru","Stationery"),
  (206,"HomeNeeds Pvt Ltd","Pune","Home Appliances"),
  (207,"National Grocers","Ahmedabad","Groceries"),
  (208,"Smart Electronics","Kolkata","Electronics"),
  (209,"Daily Essentials","Hyderabad","Personal Care"),
  (210,"Kitchen World","Chennai","Home Appliances")
  ]
  suppliers_columns = [
  "supplier_id",
  "supplier_name",
  "supplier_city",
  "specialization"
  ]
  return spark.createDataFrame(suppliers_data, suppliers_columns)

In [ ]:
@dlt.table(
    name = "bronze_products"
)

def bronze_products():
  products_data = [
  (101,"Rice Bag","Groceries","Hyderabad",1200,50),
  (102,"Wheat Flour","Groceries","Bengaluru",900,80),
  (103,"Sunflower Oil","Groceries","Mumbai",1800,40),
  (104,"Milk Pack","Dairy","Chennai",60,200),
  (105,"Cheese Block","Dairy","Delhi",450,70),
  (106,"Soap","Personal Care","Kolkata",120,300),
  (107,"Shampoo","Personal Care","Pune",320,150),
  (108,"Toothpaste","Personal Care","Ahmedabad",90,250),
  (109,"Notebook","Stationery","Hyderabad",75,500),
  (110,"Pen Pack","Stationery","Mumbai",110,400),
  (111,"LED TV","Electronics","Delhi",45000,15),
  (112,"Refrigerator","Electronics","Chennai",38000,10),
  (113,"Washing Machine","Electronics","Bengaluru",29000,12),
  (114,"Mobile Phone","Electronics","Hyderabad",25000,35),
  (115,"Laptop","Electronics","Pune",62000,18),
  (116,"Air Conditioner","Electronics","Mumbai",42000,9),
  (117,"Mixer Grinder","Home Appliances","Kolkata",3500,45),
  (118,"Water Purifier","Home Appliances","Delhi",12000,20),
  (119,"Ceiling Fan","Home Appliances","Ahmedabad",2800,60),
  (120,"Gas Stove","Home Appliances","Chennai",5500,25)
  ]
  products_columns = [
  "product_id",
  "product_name",
  "category",
  "warehouse_city",
  "price",
  "stock_quantity"
  ]
  return spark.createDataFrame(products_data, products_columns)

In [ ]:
@dlt.table(
    name= "silver_tables"
)

def silver_tables():
  orders= dlt.read("bronze_orders")
  payments= dlt.read("bronze_payments")
  suppliers= dlt.read("bronze_suppliers")
  products= dlt.read("bronze_products")

  df= products.join(orders,  "product_id", "inner")\
              .join(payments, "order_id", "inner")\
              .join(suppliers, "supplier_id", "inner")\
              .withColumn("order_date", to_date(col("order_date")))\
              .withColumn("total_revenue", col("bill_amount"))
  df= df.filter((col("quantity") > 0) & (col("bill_amount") > 0))
  return df

In [ ]:
@dlt.table(
    name= "gold_city_rev"
)

def gold_city_rev():
  df= dlt.read("silver_orders")

  return df.groupby("warehouse_city")\
           .agg(sum("total_revenue").alias("total_revenue"), count("order_id").alias("total_orders"))

In [ ]:
@dlt.table(
    name= "gold_cat_rev"
)

def gold_cat_rev():
  df = dlt.read("silver_orders")

  return df.groupby("category")\
           .agg(sum("total_revenue").alias("total_revenue"), count("order_id").alias("total_orders"))